**Standalone review repo for DeepChem genomic sequence support.**

This repo contains a DeepChem-compatible wrapper for DNABERT-2 and a tutorial notebook demonstrating usage on the Human Non-TATA Promoters benchmark. The wrapper lives at `genomics/dnabert2.py`. To run a quick smoke test (CPU, <1 min): install requirements (`pip install -r requirements.txt`), run the `SMOKE TEST` cell below. For full training you'll need a GPU and the model weights (instructions in the README). 
Tested on: deepchem==2.8.0, transformers==4.40.0, torch==2.1.0.

# Tutorial: Introduction to DNABERT2
### *From Raw DNA to State-of-the-Art Foundation Models*

This tutorial demonstrates how to use the `genomics` module to process, featurize, and model genomic sequences using DeepChem's standard MolNet architecture.

We will use the **Human Non-TATA Promoters** dataset from Genomic Benchmarks to walk through a complete research workflow.


In [ ]:
# --- ENV & INSTALL (recommend using the requirements.txt shipped in the repo) ---
# Tested with: deepchem==2.8.0, torch==2.1.0, transformers==4.40.0, genomic-benchmarks==0.1.0
!pip install -q -r requirements.txt

In [ ]:
import os
import deepchem as dc
import numpy as np
import matplotlib.pyplot as plt
from genomics.loader import load_genomic_benchmark
from genomics.featurizers import DNAKmerCountFeaturizer, DNAOneHotFeaturizer
from sklearn.ensemble import RandomForestClassifier

# Set random seeds for reproducibility
np.random.seed(42)


In [ ]:
# --- SMOKE TEST: tiny reproducible run (CPU, ~30s) ---
# This uses 32 samples only to test end-to-end API; no training time needed.
from genomics.loader import load_genomic_benchmark
from genomics.dnabert2 import DNABERT2Model
import numpy as np, torch, random, os
import deepchem as dc
np.random.seed(42); torch.manual_seed(42); random.seed(42)

# load a tiny split
tasks, datasets, _ = load_genomic_benchmark(
    dataset_name="human_nontata_promoters",
    featurizer="onehot",
    splitter="official",
    reload=False
)
# reduce to tiny subset
train = datasets[0]
test = datasets[-1]
def tiny(ds, n=32):
    X = ds.X[:n]
    y = ds.y[:n]
    return dc.data.NumpyDataset(X, y)
tiny_train = tiny(train, n=32)
tiny_test  = tiny(test, n=32)

# create minimal model (no heavy weights download or short-circuit if missing)
try:
    m = DNABERT2Model(task="classification", n_tasks=1, model_dir="tmp_check", batch_size=8, max_seq_length=64)
    m.fit(tiny_train, nb_epoch=1)
    print("smoke test fit OK")
    preds = m.predict(tiny_test)
    print("pred shape:", np.asarray(preds).shape)
except Exception as e:
    print("Smoke test failed:", e)
    raise

## Section 1: Standardized Data Loading
DeepChem's `load_genomic_benchmark` function handles downloading, splitting, and memory-efficient storage.

We use `splitter="official"` to ensure we use the benchmark's standardized Train/Test sets, which is critical for scientific reproducibility.


In [ ]:
# Load the dataset using the official benchmark splits
tasks, datasets, transformers = load_genomic_benchmark(
    dataset_name="human_nontata_promoters",
    featurizer="raw",    # Load raw sequences first
    splitter="official",
    reload=False
)

# Some benchmarks might only have Train/Test (Valid may be empty)
train, test = datasets[0], datasets[-1]

print(f"Tasks: {tasks}")
print(f"Train samples: {len(train)}")
print(f"Test samples:  {len(test)}")
print(f"\nSample DNA sequence: {train.X[0][:50]}...")


## Section 2: Flexible DNA Featurization
Genomic sequences need to be converted into numerical tensors. DeepChem makes this effortless.

1. **One-Hot Encoding**: Best for CNNs.
2. **K-mer Frequency**: Best for traditional ML and Transformers.


In [ ]:
# 1. One-Hot Featurization (Shortcut: "onehot")
_, oh_datasets, _ = load_genomic_benchmark(
    dataset_name="human_nontata_promoters",
    featurizer="onehot",
    splitter="official",
    reload=False
)
oh_train = oh_datasets[0]
print(f"One-Hot Shape: {oh_train.X.shape} (N, Length, 4)")

# 2. K-mer Featurization (Using 4-mers)
kmer_feat = DNAKmerCountFeaturizer(k=4)
_, kmer_datasets, _ = load_genomic_benchmark(
    dataset_name="human_nontata_promoters",
    featurizer=kmer_feat,
    splitter="official",
    reload=False
)
kmer_train = kmer_datasets[0]
print(f"K-mer Shape: {kmer_train.X.shape} (N, 256 frequencies)")


## Section 3: Data Visualization
Understanding the "signal" in your genomic data. Let's visualize the 4-mer frequency distribution between promoters and non-promoters.


In [ ]:
# Compare mean K-mer frequencies across classes
pos_mask = (kmer_train.y == 1).flatten()
neg_mask = (kmer_train.y == 0).flatten()

pos_mean = kmer_train.X[pos_mask].mean(axis=0)
neg_mean = kmer_train.X[neg_mask].mean(axis=0)

plt.figure(figsize=(12, 4))
plt.plot(pos_mean, label="Promoter (Positive)", alpha=0.8)
plt.plot(neg_mean, label="Non-Promoter (Negative)", alpha=0.8)
plt.fill_between(range(256), pos_mean, neg_mean, color='gray', alpha=0.2)
plt.title("4-mer Frequency Signature: Promoters vs Non-Promoters")
plt.xlabel("K-mer Index")
plt.ylabel("Mean Frequency")
plt.legend()
plt.show()


## Section 4: Baseline Modeling with Random Forest
We use DeepChem's `SklearnModel` wrapper to train a quick baseline on our K-mer features.


In [ ]:
# Initialize and train
rf = RandomForestClassifier(n_estimators=100, n_jobs=-1)
model = dc.models.SklearnModel(rf)
model.fit(kmer_train)

# Evaluate using DeepChem's ROC-AUC metric
metric = dc.metrics.Metric(dc.metrics.roc_auc_score, mode="classification")
kmer_test = kmer_datasets[-1]
test_score = model.evaluate(kmer_test, [metric])

print(f"Baseline Test ROC-AUC: {test_score['roc_auc_score']:.4f}")


## Section 5: Foundation Models (DNABERT-2)
Finally, we scale up to a Transformer-based foundation model. DNABERT-2 is pre-trained on massive genomic corpora and can be fine-tuned for specific tasks.


In [ ]:
from genomics.dnabert2 import DNABERT2Model
import deepchem as dc

metric = dc.metrics.Metric(dc.metrics.roc_auc_score, mode="classification")

model_dir = os.path.join(os.getcwd(), "dnabert2_checkpoints")
os.makedirs(model_dir, exist_ok=True)

bert_model = DNABERT2Model(
    task="classification",
    n_tasks=1,
    model_dir=model_dir,
    batch_size=4,
    max_seq_length=128,
    learning_rate=2e-5,
)

bert_model.fit(train, nb_epoch=1)
bert_score = bert_model.evaluate(test, [metric], n_classes=2)
print(f"DNABERT-2 Test ROC-AUC: {bert_score[metric.name]:.4f}")

In [ ]:
metrics = [
    dc.metrics.Metric(dc.metrics.roc_auc_score, mode="classification"),
]
scores = bert_model.evaluate(test, metrics, n_classes=2)
for name, value in scores.items():
    print(f"  {name}: {value:.4f}")

In [ ]:
import numpy as np
from sklearn.metrics import recall_score

# Raw predictions: shape (n, 1, 2) — logits or probs
preds = bert_model.predict(test)

# Class indices: (n,) with values 0 or 1
y_pred = np.argmax(preds, axis=-1).squeeze()

# Labels: (n,) — match loader's (n, 1) layout
y_true = test.y.squeeze()

recall = recall_score(y_true, y_pred)
print(f"Recall: {recall:.4f}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Let's visualize the performance comparison between the Random Forest baseline and DNABERT-2
models = ['Random Forest (4-mer)', 'DNABERT-2 (Raw Seq)']
roc_aucs = [test_score['roc_auc_score'], bert_score[metric.name]]

plt.figure(figsize=(8, 5))
sns.barplot(x=models, y=roc_aucs, palette="viridis")
plt.title("Model Comparison: Human Non-TATA Promoters (ROC-AUC)")
plt.ylabel("Test ROC-AUC")
plt.ylim(0, 1.0)

# Add text labels on top of bars
for i, v in enumerate(roc_aucs):
    plt.text(i, v + 0.02, f"{v:.4f}", ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

## Interpretation of Results

1. **Random Forest vs. DNABERT-2**: As expected, the Random Forest baseline on k-mer frequencies provides a surprisingly strong baseline for sequence classification. However, the pre-trained foundation model parameterizes more effectively from raw sequences directly, matching or out-performing the baseline after minimal fine-tuning.
2. **Representation differences**: The baseline model relies on global counts of 4-mers, lacking structural and position-specific awareness. DNABERT-2 uses BPE tokenization and multi-head self-attention to leverage both positional and contextually aware embeddings of bases, giving it a capacity for much more complex genomics understanding.
3. **DeepChem API Compatibility**: Most importantly, both workflows fit seamlessly into the `<model>.fit(train)` -> `<model>.evaluate(test)` API, using standard `dc.metrics` tools. This shows how DeepChem allows effortless interchangeability between classical ML on engineered features and state-of-the-art transformer models on raw datasets.

## References

- **Genomic Benchmarks**: [ML-Bioinfo-CEITEC/genomic_benchmarks](https://github.com/ML-Bioinfo-CEITEC/genomic_benchmarks)
- **DNABERT-2**: Zhou, Z. et al. DNABERT-2: Efficient Foundation Model and Benchmark for Multi-Species Genome. arXiv:2306.15006 (2023)
- **DeepChem**: [deepchem/deepchem](https://github.com/deepchem/deepchem)